---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — include the notebook name, cell number, and what went wrong.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first.*

### 💾 Save your own copy
> **File → Save a copy in Drive** — saves a personal editable copy to your Google Drive.
> Do this before making edits, otherwise changes are lost when the session ends.

# 🔵 K-Nearest Neighbors
**ISLP Chapter 2 · Pattern Recognition for the Rest of Us**

> The most intuitive ML model: to classify a new point, find its K nearest neighbours in the training data and take a majority vote. No equations to fit — just distance and memory.

### What you'll learn
- How KNN makes predictions step by step
- Why feature scaling is critical for distance-based models
- Choosing K with cross-validation
- The curse of dimensionality

### Time: ~45 minutes

In [ ]:
# WARNING Run this cell first - sets up data and imports
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import accuracy_score, classification_report

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

# Load UCI Adult Income - real data (48,842 records)
# Source: https://archive.ics.uci.edu/dataset/2/adult

_UCI_COLS = ['age','workclass','fnlwgt','education','education_num',
             'marital_status','occupation','relationship','race','sex',
             'capital_gain','capital_loss','hours_per_week','native_country','income']

def _build_adult(df):
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    df = df.drop(columns=[c for c in df.columns if 'fnlwgt' in c], errors='ignore')
    df = df.replace(' ?', np.nan).dropna()
    if 'class' in df.columns and 'income' not in df.columns:
        df = df.rename(columns={'class': 'income'})
    df['income_bin'] = (df['income'].astype(str).str.strip()
                                    .str.replace('.','',regex=False)
                                    .eq('>50K').astype(int))
    return df

_loaded = False
for _attempt in ['repo', 'uci_raw', 'openml']:
    try:
        if _attempt == 'repo':
            adult = pd.read_csv('https://raw.githubusercontent.com/ladataanalytics/pattern-recognition-notebooks/main/data/adult.csv')
            if 'income_bin' not in adult.columns:
                adult = _build_adult(adult)
            _src = 'repo CSV (real UCI data)'
        elif _attempt == 'uci_raw':
            _tr = pd.read_csv(
                'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data',
                header=None, names=_UCI_COLS, na_values=' ?', skipinitialspace=True)
            _te = pd.read_csv(
                'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test',
                header=None, names=_UCI_COLS, na_values=' ?',
                skipinitialspace=True, skiprows=1)
            _te['income'] = _te['income'].str.replace('.','',regex=False)
            adult = _build_adult(pd.concat([_tr, _te], ignore_index=True))
            _src = 'UCI raw CSV'
        elif _attempt == 'openml':
            from sklearn.datasets import fetch_openml
            adult = _build_adult(
                fetch_openml(data_id=1590, as_frame=True, parser='auto').frame.copy())
            _src = 'OpenML (id=1590)'
        _loaded = True
        break
    except Exception:
        continue

if not _loaded:
    raise RuntimeError('Could not load Adult dataset - check internet connection.')

# Subsample to 15k for KNN speed (KNN is O(n) at predict time)
if len(adult) > 15000:
    adult = (adult.groupby('income_bin', group_keys=False)
                  .apply(lambda g: g.sample(
                      int(15000 * len(g)/len(adult)+0.5), random_state=42))
                  .reset_index(drop=True))
    _src += f' - stratified {len(adult):,} rows'

print(f'Dataset ({_src}): {adult.shape}')
print(f'  Income >$50K: {adult["income_bin"].mean():.1%}  (full UCI: 23.9%)')
print(f'  capital_gain zeros: {(adult["capital_gain"]==0).mean():.1%}')
print(f'  Python {sys.version.split()[0]} | pandas {pd.__version__}')
print('Setup complete')


## 📐 Part 1 — How KNN Works

Given a new unlabelled point:
1. Compute distance to every training point (usually Euclidean)
2. Find the K closest points (the "neighbours")
3. Take a majority vote of their labels → prediction

**Key parameters:**
- **K** — number of neighbours. Small K = complex boundary; large K = smooth boundary
- **Distance metric** — Euclidean by default; Manhattan and Chebyshev also common
- **Feature scaling** — mandatory: unscaled features with different ranges dominate distance

**When KNN works well:** small-to-medium datasets, non-linear boundaries, interpretability matters
**When it struggles:** high dimensions, large datasets (slow at prediction), imbalanced classes

In [ ]:
# Prepare features and understand the scaling effect
features = ['age','education_num','hours_per_week','capital_gain','capital_loss']
X = adult[features].values.astype(float)
y = adult['income_bin'].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# Standard scaling
scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_te_s = scaler.transform(X_te)

# Log-transform capital_gain and capital_loss before scaling
# Both are zero-inflated (capital_gain: 91.7% zeros, max=99,999)
# log1p compresses extreme values: log1p(99999) = 11.5
X_log = X.copy()
X_log[:, 3] = np.log1p(X_log[:, 3])  # capital_gain
X_log[:, 4] = np.log1p(X_log[:, 4])  # capital_loss
X_tr_l, X_te_l, _, _ = train_test_split(
    X_log, y, test_size=0.25, random_state=42, stratify=y)
scaler_l = StandardScaler().fit(X_tr_l)
X_tr_ls  = scaler_l.transform(X_tr_l)
X_te_ls  = scaler_l.transform(X_te_l)

# Compare all three
print(f'capital_gain: {(X[:,3]==0).mean():.0%} zeros, max={int(X[:,3].max()):,}')
print(f'After log1p:  max becomes {np.log1p(X[:,3].max()):.1f}')
print()
print(f'  {"K":>4}  {"Raw":>8}  {"Std scaled":>11}  {"Log+scaled":>11}  Best')
print('-' * 52)
best_counts = {'Raw':0, 'Std scaled':0, 'Log+scaled':0}
for K in [1, 3, 5, 11, 21, 51]:
    raw = KNeighborsClassifier(K).fit(X_tr,    y_tr).score(X_te,    y_te)
    std = KNeighborsClassifier(K).fit(X_tr_s,  y_tr).score(X_te_s,  y_te)
    log = KNeighborsClassifier(K).fit(X_tr_ls, y_tr).score(X_te_ls, y_te)
    best = max({'Raw':raw,'Std scaled':std,'Log+scaled':log},
               key={'Raw':raw,'Std scaled':std,'Log+scaled':log}.get)
    best_counts[best] += 1
    print(f'  {K:>4}  {raw:>8.3f}  {std:>11.3f}  {log:>11.3f}  {best}')
print()
print(f'Winner tally: {best_counts}')
print()
print('Why raw sometimes beats standard scaling:')
print('  capital_gain is zero for ~92% of records')
print('  StandardScaler uses the full std including outliers up to 99,999')
print('  After scaling, the rare large values are amplified -- hurting KNN')
print('  Raw KNN ignores capital_gain most of the time because it is 0')
print()
print('log1p fix: compresses skewed distributions without breaking zeros')
print('  log1p(0)     = 0   (zeros stay zero)')
print('  log1p(99999) = 11.5 (outliers compressed)')
print()
print('Key lesson: for KNN the correct pipeline is:')
print('  1. Transform skewed/zero-inflated features with log1p or sqrt')
print('  2. Then StandardScaler so all features share the same distance scale')
print()
print('Using log+scaled features for all subsequent cells.')

# Use log+scaled for the rest of the notebook
X_tr_s = X_tr_ls
X_te_s = X_te_ls
scaler = scaler_l


## 🔬 Part 2 — Decision Boundaries (K=1 to K=51)

Using 2 features (age, education) so we can visualise the boundary in 2D.

- **K=1** — every training point is its own region: jagged, overfits
- **K=51** — large smooth regions: may miss real structure
- CV accuracy increases with K until it plateaus — the sweet spot

In [ ]:
# Decision boundary visualisation (2 features for plotting)
# Separate 2-feature scaler -- the main scaler was fit on 5 features

# Apply same log1p transform to capital_gain proxy (education_num is fine as-is)
scaler_2d = StandardScaler().fit(
    adult[['age','education_num']].values)
X_2d = scaler_2d.transform(adult[['age','education_num']].values)
y_2d = adult['income_bin'].values

# Subsample for scatter
rng  = np.random.default_rng(0)
idx  = rng.choice(len(X_2d), size=min(2000, len(X_2d)), replace=False)
Xp, yp = X_2d[idx], y_2d[idx]

x0, x1 = X_2d[:,0].min()-0.3, X_2d[:,0].max()+0.3
y0, y1 = X_2d[:,1].min()-0.3, X_2d[:,1].max()+0.3
xx, yy = np.meshgrid(np.linspace(x0, x1, 300),
                     np.linspace(y0, y1, 300))
grid = np.c_[xx.ravel(), yy.ravel()]

K_vals = [1, 5, 15, 51]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

for ax, K in zip(axes, K_vals):
    knn = KNeighborsClassifier(n_neighbors=K).fit(X_2d, y_2d)
    Z   = knn.predict(grid).reshape(xx.shape)

    # Light fills: blue = predict <=50K, orange = predict >50K
    ax.contourf(xx, yy, Z, levels=[-0.5, 0.5, 1.5],
                colors=['#c8daef','#f7d8c0'], alpha=0.6)
    # Thin boundary line
    ax.contour(xx, yy, Z, levels=[0.5],
               colors=['#555555'], linewidths=0.8)

    # Scatter: income<=50K blue, >50K orange
    ax.scatter(Xp[yp==0,0], Xp[yp==0,1], c='#2166ac',
               alpha=0.25, s=6, zorder=3, label='<=50K')
    ax.scatter(Xp[yp==1,0], Xp[yp==1,1], c='#d6604d',
               alpha=0.35, s=6, zorder=3, label='>50K')

    cv = cross_val_score(KNeighborsClassifier(K), X_2d, y_2d,
                         cv=5, n_jobs=-1).mean()
    ax.set_title(f'K = {K}\n5-fold CV = {cv:.3f}', fontsize=10)
    ax.set_xlabel('Age (z-score)')
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    if K == 1:
        ax.set_ylabel('Education years (z-score)')
        ax.legend(fontsize=8, markerscale=2, loc='lower right')

plt.suptitle(
    'KNN Decision Boundaries — Adult Income\n'
    'Blue region = predict <=50K   |   Orange region = predict >50K',
    fontsize=11)
plt.tight_layout()
plt.show()

print('Horizontal banding = education_num is a discrete integer (1-16).')
print('Real data structure, not a plotting artefact.')
print()
print('K=1: jagged boundary, memorises every training point -- overfits.')
print('K=51: smooth boundary, CV accuracy is highest in this range.')
print()
print('The bias-variance U-curve: CV rises as K grows here because')
print('we have not yet crossed the underfitting threshold.')
print('To see it degrade, run the CV curve in Part 3 -- that extends')
print('to K=50 and shows where the curve flattens or turns.')
print('K=51 is not underfitting; it is near-optimal for these 2 features.')


#Example prediction and 10 nearest neighbors

In [ ]:
rng = np.random.default_rng(42)
idx_to_predict = rng.choice(len(X_te_s))

single_point_scaled = X_te_s[idx_to_predict]
single_point_raw = X_te[idx_to_predict]
true_label = y_te[idx_to_predict]

# Train KNN with K=10 on the scaled training data
knn_example = KNeighborsClassifier(n_neighbors=10).fit(X_tr_s, y_tr)

# Find the 10 nearest neighbors for the single point
distances, neighbor_indices = knn_example.kneighbors(single_point_scaled.reshape(1, -1))

# Get the labels and raw features of these neighbors
neighbor_labels = y_tr[neighbor_indices[0]]
neighbor_raw_features = X_tr[neighbor_indices[0]]

print(f"Selected test point index: {idx_to_predict}")
print(f"True label: {true_label} (0: <=50K, 1: >50K)")

print("\n--- Test Point Details ---")
print("Features (raw values):")
for i, feature_name in enumerate(features):
    print(f"  {feature_name}: {np.round(single_point_raw[i], 2)}")

print("Features (scaled values):")
for i, feature_name in enumerate(features):
    print(f"  {feature_name}: {np.round(single_point_scaled[i], 2)}")

print("\n--- 10 Nearest Neighbors Details ---")
for i, (label, raw_features) in enumerate(zip(neighbor_labels, neighbor_raw_features)):
    print(f"Neighbor {i+1}:")
    print(f"  Label: {label}")
    print("  Raw Features:")
    for j, feature_name in enumerate(features):
        print(f"    {feature_name}: {np.round(raw_features[j], 2)}")

# Make a prediction for the single point
prediction = knn_example.predict(single_point_scaled.reshape(1, -1))[0]

print(f"\nKNN prediction for this point: {prediction} (0: <=50K, 1: >50K)")

# Determine if the prediction is correct
if prediction == true_label:
    print("Prediction is CORRECT.")
else:
    print("Prediction is INCORRECT.")


## 📊 Part 3 — Choosing K with Cross-Validation

We can't use the test set to choose K — that would be data leakage.
Instead, use **k-fold cross-validation on the training set** to estimate generalisation error for each K, then pick the best.

In [ ]:
# ── CV curve: accuracy vs K ──────────────────────────────────────────────────
# Use the scaled FULL dataset (X_tr_s / X_te_s defined in Part 1)
K_range   = range(1, 51)
cv_scores = []

for K in K_range:
    scores = cross_val_score(
        KNeighborsClassifier(n_neighbors=K, metric='euclidean'),
        X_tr_s, y_tr, cv=10, scoring='accuracy', n_jobs=-1)
    cv_scores.append(scores.mean())

best_K = list(K_range)[cv_scores.index(max(cv_scores))]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(list(K_range), cv_scores, 'o-', color='#1e5fa8',
        lw=1.5, markersize=4, label='10-fold CV accuracy')
ax.axvline(best_K, color='#e85d20', lw=2, ls='--',
           label=f'Best K = {best_K}  (acc = {max(cv_scores):.3f})')
ax.set_xlabel('K (number of neighbours)')
ax.set_ylabel('10-Fold CV Accuracy')
ax.set_title('Choosing K via Cross-Validation — Adult Income')
ax.legend()
plt.tight_layout()
plt.show()

# Evaluate best K on held-out test set
knn_best = KNeighborsClassifier(n_neighbors=best_K).fit(X_tr_s, y_tr)
y_pred   = knn_best.predict(X_te_s)

print(f"K=1  train acc : {KNeighborsClassifier(1).fit(X_tr_s,y_tr).score(X_tr_s,y_tr):.3f}  (memorises)")
print(f"K=1  test  acc : {KNeighborsClassifier(1).fit(X_tr_s,y_tr).score(X_te_s,y_te):.3f}  (overfits)")
print(f"K={best_K} test  acc : {knn_best.score(X_te_s, y_te):.3f}  (best generalisation)")
print(f"\n=== Classification Report (K={best_K}) ===")
print(classification_report(y_te, y_pred, target_names=['≤50K', '>50K']))


## ✅ Part 4 — Curse of Dimensionality

As the number of features grows, the volume of the feature space explodes. Training points become increasingly sparse and **all distances converge** — your nearest neighbour is barely closer than a random point. KNN relies entirely on meaningful distance, so it breaks down in high dimensions.

**Rule of thumb:** KNN degrades noticeably beyond ~10–20 features unless the data is very dense.

In [ ]:
# ── Accuracy vs dimensions (real + noise features) ───────────────────────────
from sklearn.metrics import pairwise_distances

np.random.seed(42)
base_X  = StandardScaler().fit_transform(
    adult[features].values)          # 5 real features, already scaled
base_y  = adult['income_bin'].values

dim_list    = [2, 5, 10, 20, 50, 100]
acc_list    = []
ratio_list  = []   # std/mean of pairwise distances — measures concentration

for d in dim_list:
    if d <= 5:
        Xd = base_X[:, :d]
    else:
        noise = np.random.randn(len(base_X), d - 5)
        Xd    = np.hstack([base_X, noise])

    # Accuracy
    acc = cross_val_score(KNeighborsClassifier(15), Xd, base_y,
                          cv=5, n_jobs=-1).mean()
    acc_list.append(acc)

    # Distance concentration on a small sample
    Xs = Xd[:500]
    D  = pairwise_distances(Xs[:200], Xs[200:400]).ravel()
    ratio_list.append(D.std() / D.mean())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(dim_list, acc_list, 'o-', color='#e85d20', lw=2, markersize=7)
axes[0].axvline(5, color='#1e5fa8', ls='--', lw=1.5, alpha=0.7,
                label='Real features end')
axes[0].set_xlabel('Total features (5 real + noise)')
axes[0].set_ylabel('5-Fold CV Accuracy')
axes[0].set_title('KNN Accuracy Degrades with Noise Dimensions')
axes[0].legend()

axes[1].plot(dim_list, ratio_list, 's-', color='#1e5fa8', lw=2, markersize=7)
axes[1].set_xlabel('Number of Dimensions')
axes[1].set_ylabel('Std(dist) / Mean(dist)')
axes[1].set_title('Distance Concentration\n(lower = neighbours barely closer than random)')

plt.tight_layout()
plt.show()

print("\n📌 5 real features → accuracy peaks, then falls as noise is added.")
print("   Distance concentration: in 100D, std/mean → near 0 —")
print("   all neighbours are equally distant. KNN becomes random guessing.")
print("   Solution: feature selection, PCA, or switch to a tree-based model.")


## 💼 Exercise

Using the Adult Income dataset loaded in the setup cell:

**Task 1 — Categorical features:** Encode `sex` and `workclass` as binary/dummy variables and add them to the 5 numeric features. Retrain KNN with the best K from Part 3. Does adding categorical features improve test accuracy?

**Task 2 — Distance metrics:** Compare `euclidean`, `manhattan`, and `chebyshev` at K=15 using 5-fold CV on the scaled training set. Which metric performs best?

**Task 3 — KNN for regression:** Use `KNeighborsRegressor` to predict `hours_per_week` from age, education_num, and capital_gain. Report test RMSE for K=5, 15, and 51.

In [ ]:
# ── Exercise workspace ───────────────────────────────────────────────────────
# Variables available: adult, X_tr_s, X_te_s, y_tr, y_te, scaler, features

# Task 1 — Add categorical features

# Task 2 — Compare distance metrics
from sklearn.model_selection import cross_val_score
for metric in ['euclidean', 'manhattan', 'chebyshev']:
    pass  # replace with your code

# Task 3 — KNN Regression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error


In [ ]:
# @title 📝 Quiz — K-Nearest Neighbors
# @markdown Answer each question, then run this cell.

# @markdown **Q1:** How does KNN predict the class of a new point?
# @markdown **Q2:** Why must features be standardized before applying KNN?
# @markdown **Q3:** What is the bias-variance tradeoff as K increases from 1 to N?
# @markdown **Q4:** What is the curse of dimensionality and how does it affect KNN?
# @markdown **Q5:** Name two situations where you would choose a different model over KNN.

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the submission cell for AI feedback")


In [ ]:
_NB_NAME_ = "K-Nearest Neighbors"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f'Please grade my quiz answers for the "{_NB_TITLE}" notebook')
    print(f'from "Data Pattern Recognition for the Rest of Us" (based on ISLP).')
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why")
    print("  3. Give the ideal complete answer")
    print("  4. If wrong or partial, say which Part to revisit")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan")


---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md)

---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*